# CHB-MIT 单通道预测 + 六通道教师蒸馏（alpha 扫描）

`colab/` 文件夹放在 MyDrive 根目录，把这个笔记本上传到 Colab，挂载后按顺序跑。

- 代码：`/content/drive/MyDrive/colab`
- EDF：`/content/chbmit`（跑完一个患者删一个）
- 结果：`/content/drive/MyDrive/colab/results/{pid}/`

每个患者：LOOCV 折 → 先训六通道教师（ictal vs 间期，1:1）→ 冻结 → 对每个 alpha 训六个单电极学生 → 用验证集概率训电极融合模型 → 测试。
归一化每患者每折一套，teacher 和 student 共用，统计池排除两边的 val/test 窗。

In [ ]:
!pip install -q mne

In [ ]:
import gc
import json
import shutil
import sys
from datetime import datetime
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

CODE = Path("/content/drive/MyDrive/colab")
DATA = Path("/content/chbmit")
OUT = CODE / "results_distill"

sys.path.insert(0, str(CODE))
DATA.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import tools as t
import colab_tools as ct
from models import ConvBILSTMClassify, Detection

print("device =", t.DEVICE)

In [ ]:
ELECS = ["Fp1-F7", "Fp2-F8", "T7-P7", "T8-P8", "P7-O1", "P8-O2"]
PATIENTS = ["chb%02d" % i for i in range(1, 25)]
# PATIENTS = ["chb01"]                # 调试时只跑一个

SPH, SOP = 2 * 60, 30 * 60           # 发作前留白 2 min，发作前期 30 min
INTER_THR = 2 * 60 * 60              # 发作间期离发作至少 2 h
WIN_LEN, WIN_OL = 4, 2               # 4 s 窗口，50% 重叠
ENC_VAL, CLS_VAL = 0.1, 0.1
SEED = 0

ALPHAS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]   # FGL: alpha*CE + (1-alpha)*KL，1.0 是无蒸馏对照
T = 2

CHAN_ATTEN, TEMP_ATTEN = False, True
HIDDEN, DROPOUT = 20, 0.7
LR_T, LR_S, WD = 1e-3, 1e-4, 1e-3
BATCH = 32
EPOCHS_T, PATIENCE_T = 100, 15
EPOCHS_S, PATIENCE_S = 30, 10        # 跟 261105 的单电极口径一致，方便对比

CONFIG = dict(
    alphas=ALPHAS, T=T, learning_rate=LR_S, weight_decay=WD, batch_size=BATCH,
    dropout=DROPOUT, hidden_size=HIDDEN, channal_attention=CHAN_ATTEN,
    temporal_attention=TEMP_ATTEN, epochs_s=EPOCHS_S, patience_s=PATIENCE_S,
    epochs_t=EPOCHS_T, patience_t=PATIENCE_T,
)

## 主流程：下载一个 → 训教师 → 扫 alpha 训学生 → 融合 → 测试

1. `ct.download_patient` 下 EDF
2. `t.preprocess` 切窗打标签
3. `ct.student_dataset` 按 LOOCV 建学生张量（不建自编码器那份大张量）
4. `t.teacher_from_student` 教师的折向学生看齐，ictal 与间期 1:1
5. `t.fold_norm_stats` 每折一套 mean/std，teacher 与 student 共用
6. 每折训一次教师，冻结；对每个 alpha 训六个学生
7. 每折每 alpha 用验证集概率训 `ElecAfterMultiHeadAttention` 融合，测测试集
8. 结果存 Drive，删本地 EDF

In [ ]:
torch.backends.cudnn.benchmark = True   # 固定形状，让 cudnn 选最快的卷积算法


def distill_loss(student_logits, y_student, teacher_logits, alpha, T):
    """FGL：alpha*CE(student, y) + (1-alpha)*T^2*BCE(student/T, teacher/T)
    teacher_logits 是冻结后预先算好的常量，不用每步再前向
    """
    ce = F.binary_cross_entropy_with_logits(student_logits.squeeze(-1), y_student.squeeze(-1))
    with torch.no_grad():
        p_t = torch.sigmoid(teacher_logits / T).squeeze(-1)
    distill = F.binary_cross_entropy_with_logits((student_logits / T).squeeze(-1), p_t) * T ** 2
    return alpha * ce + (1 - alpha) * distill


def make_batches(*ts, shuffle=False, batch_size=32):
    """张量已经在 GPU 上，直接按索引切片，省掉 DataLoader 的逐样本 collate 和 H2D 拷贝"""
    n = len(ts[0])
    order = torch.randperm(n, device=ts[0].device) if shuffle else torch.arange(n, device=ts[0].device)
    for i in range(0, n, batch_size):
        idx = order[i:i + batch_size]
        yield tuple(x[idx] for x in ts)


def predict(model, x, y, batch_size=BATCH):
    probs, targets = [], []
    model.eval()
    with torch.no_grad():
        for bx, by in make_batches(x, y, batch_size=batch_size):
            probs.append(torch.sigmoid(model(bx)).ravel())
            targets.append(by.ravel())
    return torch.cat(probs).cpu().numpy(), torch.cat(targets).cpu().numpy()


In [ ]:
for pid in PATIENTS:
    done = OUT / pid / ("%s.json" % pid)
    if done.exists():
        print("[跳过] 已完成", pid)
        continue
    print("=" * 15, pid, "=" * 15)
    out = OUT / pid
    weight_dir = out / "weight"
    weight_dir.mkdir(parents=True, exist_ok=True)
    now = datetime.now().strftime("%y%m_%d_%H%M%S")

    # 1. 下载 + 预处理
    pdir = ct.download_patient(pid, DATA)
    windows, meta = t.preprocess(str(pdir), SPH, SOP, INTER_THR, WIN_LEN, WIN_OL, ELECS)

    # 2. 学生折 + 教师折（对齐）+ 统一归一化参数
    student_ds = ct.student_dataset(ELECS, meta, SEED, (ENC_VAL, CLS_VAL))
    teacher_ds = t.teacher_from_student(ELECS, meta, windows, student_ds[0], SEED)
    keep = {d["seizure_id"] for d in teacher_ds}
    student_ds = [[f for f in elec if f["seizure_id"] in keep] for elec in student_ds]
    norm_stats = [t.fold_norm_stats(meta, windows, ELECS, s, d)
                  for s, d in zip(student_ds[0], teacher_ds)]
    print("[%s] %d 折 %s" % (pid, len(teacher_ds), [d["seizure_id"] for d in teacher_ds]))

    data = {"config": CONFIG, "teacher": {}, "students": {}, "test_index": []}

    for fold, d in enumerate(teacher_ds):
        mean, std = norm_stats[fold]
        print("---- fold %d/%d seizure %s ----" % (fold + 1, len(teacher_ds), d["seizure_id"]))

        # 3. 教师：六通道 ictal vs 间期（张量直接放 GPU）
        t_tr = (t.norm_wins(d["train_wins"], mean, std).to(t.DEVICE), d["y_train"].to(t.DEVICE))
        t_va = (t.norm_wins(d["val_wins"], mean, std).to(t.DEVICE), d["y_val"].to(t.DEVICE))
        torch.manual_seed(SEED * 1000 + fold)      # 教师初始化、打乱可复现
        teacher = Detection().to(t.DEVICE)
        opt_t = torch.optim.Adam(teacher.parameters(), lr=LR_T)
        best_val, best_state, counter = float("inf"), {}, 0
        for epoch in range(EPOCHS_T):
            teacher.train()
            for x, y in make_batches(*t_tr, shuffle=True, batch_size=BATCH):
                opt_t.zero_grad()
                loss = F.binary_cross_entropy_with_logits(teacher(x), y)
                loss.backward()
                opt_t.step()
            teacher.eval()
            val_loss, nb = 0, 0
            with torch.no_grad():
                for x, y in make_batches(*t_va, batch_size=BATCH):
                    val_loss += F.binary_cross_entropy_with_logits(teacher(x), y).item()
                    nb += 1
            val_loss /= nb
            if val_loss < best_val:
                best_val, counter = val_loss, 0
                best_state = {k: v.cpu().clone() for k, v in teacher.state_dict().items()}
            else:
                counter += 1
                if counter >= PATIENCE_T:
                    break
        teacher.load_state_dict(best_state)
        teacher.eval()
        teacher.requires_grad_(False)
        t_path = weight_dir / ("%s_teacher_fold%d.pth" % (now, fold + 1))
        torch.save(best_state, t_path)
        data["teacher"]["fold_%d" % (fold + 1)] = {"weight": str(t_path), "val_loss": best_val}
        del t_tr, t_va

        # 4. 教师冻结后，软标签每折只算一遍：6 alpha × 6 电极共用，省掉 36 倍教师前向
        s0 = student_ds[0][fold]
        x6 = {key: t.norm_wins(torch.tensor(
                  np.stack([windows[en][s0[key + "_idx"]] for en in ELECS], axis=1),
                  dtype=torch.float32), mean, std).to(t.DEVICE)
              for key in ("train", "val", "test")}
        with torch.no_grad():
            t_log = {key: teacher(x6[key]) for key in ("train", "val", "test")}
        del x6
        torch.cuda.empty_cache()

        # 5. 学生输入：每折每电极标准化一次并放到 GPU，六个 alpha 复用
        sw = []
        for e, elec in enumerate(ELECS):
            s = student_ds[e][fold]
            dd = {}
            for key in ("train", "val", "test"):
                idx = s[key + "_idx"]
                dd[key + "_wins"] = t.norm_wins(
                    torch.tensor(windows[elec][idx][:, None, :], dtype=torch.float32),
                    mean[e:e + 1], std[e:e + 1]).to(t.DEVICE)
                dd["y_" + key] = torch.tensor(meta[elec].iloc[idx].label.to_numpy(),
                                              dtype=torch.float32).unsqueeze(1).to(t.DEVICE)
            sw.append(dd)

        for alpha in ALPHAS:
            val_probs, test_probs, elec_metrics = [], [], []
            vt = tt = None
            for e, elec in enumerate(ELECS):
                s = sw[e]
                # 同一 (患者, 折, 电极) 用同一个种子：六个 alpha 的初始化/打乱完全相同，
                # 结果差异只来自 alpha，否则扫 alpha 被随机初始化噪声淹没
                torch.manual_seed(SEED * 1000 + fold * 10 + e)
                student = ConvBILSTMClassify(channal_attention=CHAN_ATTEN, temporal_attention=TEMP_ATTEN,
                                             dropout=DROPOUT, hidden_size=HIDDEN).to(t.DEVICE)
                opt_s = torch.optim.Adam(student.parameters(), lr=LR_S, weight_decay=WD)
                best_val, best_state, counter = float("inf"), {}, 0
                for epoch in range(EPOCHS_S):
                    student.train()
                    for x, tl, y in make_batches(s["train_wins"], t_log["train"], s["y_train"],
                                                 shuffle=True, batch_size=BATCH):
                        opt_s.zero_grad()
                        loss = distill_loss(student(x), y, tl, alpha, T)
                        loss.backward()
                        opt_s.step()
                    student.eval()
                    val_loss, nb = 0, 0
                    with torch.no_grad():
                        for x, tl, y in make_batches(s["val_wins"], t_log["val"], s["y_val"],
                                                     batch_size=BATCH):
                            val_loss += distill_loss(student(x), y, tl, alpha, T).item()
                            nb += 1
                    val_loss /= nb
                    if val_loss < best_val:
                        best_val, counter = val_loss, 0
                        best_state = {k: v.cpu().clone() for k, v in student.state_dict().items()}
                    else:
                        counter += 1
                        if counter >= PATIENCE_S:
                            break

                student.load_state_dict(best_state)
                s_path = weight_dir / ("%s_a%.1f_fold%d_%s.pth" % (now, alpha, fold + 1, elec))
                torch.save(best_state, s_path)
                data["students"].setdefault("a%.1f" % alpha, {}).setdefault("fold_%d" % (fold + 1), {})[elec] = str(s_path)

                vp, vt = predict(student, s["val_wins"], s["y_val"])
                tp, tt = predict(student, s["test_wins"], s["y_test"])
                val_probs.append(vp)
                test_probs.append(tp)
                elec_metrics.append(dict(elec=elec, **ct.classification_metrics(tt, tp)))
                del student, opt_s

            # 6. 电极融合：验证集概率拟合，测试集上用
            fusion = ct.train_fusion(val_probs, vt)
            x_te = torch.tensor(np.array(test_probs).T, dtype=torch.float32).unsqueeze(-1).to(t.DEVICE)
            with torch.no_grad():
                fused = fusion(x_te).cpu().numpy().ravel()
            m_fuse = ct.classification_metrics(tt, fused)
            data["test_index"].append(dict(fold=fold, seizure_id=int(d["seizure_id"]), alpha=alpha,
                                           metrics=elec_metrics, metrics_atten=[m_fuse]))
            ms = {k: np.mean([m[k] for m in elec_metrics])
                  for k in ("auc", "accuracy", "sensitivity", "specificity")}
            print("  alpha %.1f | 单电极 AUC %.4f ACC %.4f SEN %.4f SPE %.4f"
                  " | 融合 AUC %.4f ACC %.4f SEN %.4f SPE %.4f"
                  % (alpha, ms["auc"], ms["accuracy"], ms["sensitivity"], ms["specificity"],
                     m_fuse["auc"], m_fuse["accuracy"], m_fuse["sensitivity"], m_fuse["specificity"]))
            del fusion, x_te

        del t_log, teacher, sw
        torch.cuda.empty_cache()
        gc.collect()

    with open(out / ("%s.json" % pid), "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)
    del student_ds, teacher_ds, norm_stats, windows, meta, data
    torch.cuda.empty_cache()
    gc.collect()
    ct.clear_patient(pid, DATA)
    print("[完成]", pid)


## 汇总

In [ ]:
df = ct.summarize_distill(OUT)
print(df.groupby("alpha")[["auc", "auc_atten"]].mean())